In [1]:
import sys, os, tempfile, timeit, pickle, inspect
from dsc.dsc_io import load_dsc as __load_dsc__, source_dirs as __source_dirs__
import numpy as np

In [2]:
simres = __load_dsc__(['/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/blockdiag_p/blockdiag_p_1.pkl','/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/truncated_svd/blockdiag_p_1_nnm_sparse_1_truncated_svd_1.pkl'])

In [3]:
simres.keys()

dict_keys(['Z', 'Zmask', 'effect_size_obs', 'effect_size_true', 'Ltrue', 'Ftrue', 'Mtrue', 'Ctrue', 'nsample', 'DSC_DEBUG', 'L_est', 'F_est', 'S2'])

In [4]:
F = simres['F_est']
Ftrue = simres['Ftrue']
L = simres['L_est']
Ltrue = simres['Ltrue']
labels = simres['Ctrue']
Ztrue = simres['Z']

In [42]:
import numpy as np
from scipy.spatial import procrustes
from sklearn.cluster import AgglomerativeClustering
from sklearn import metrics as skmetrics


def standardize(X, axis = 0, center = True, scale = True):
    if center:
        X = X - np.mean(X, axis = axis, keepdims = True)
    if scale:
        X /= np.std(X, axis = axis, keepdims = True)
    return X


def mean_squared_error(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    n = np.sum(mask)
    mse = np.sum(np.square((original - recovered) * mask)) / n
    return mse


def root_mean_squared_error(original, recovered, mask = None):
    mse = mean_squared_error(original, recovered, mask = mask)
    return np.sqrt(mse)


def peak_signal_to_noise_ratio(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    omax = np.max(original[mask == 1])
    omin = np.min(original[mask == 1])
    maxsig2 = np.square(omax - omin)
    mse = mean_squared_error(original, recovered, mask)
    res = 10 * np.log10(maxsig2 / mse)
    return res


def matrix_dissimilarity_scores(original, recovered, mask = None, match = 'zerofill'):
    '''
    Procrustes analysis returns the square of the Frobenius norm.
    Use the rotated matrix to obtain the peak signal-to-noise ratio (PSNR).
    Input matrices can have different dimensions.
    There are two ways to match:
        - clip: remove information from the larger matrix
        - zerofill: pad zero columns in the smaller matrix
    '''
    n_orig = original.shape[1]
    n_recv = recovered.shape[1]
    m = original.shape[0]
    if match == 'clip':
        n = min(n_orig, n_recv)
        X = original[:, :n]
        Y = recovered[:, :n]
    elif match == 'zerofill':
        n = max(n_orig, n_recv)
        X = np.zeros((m, n))
        Y = np.zeros((m, n))
        X[:, :n_orig] = original
        Y[:, :n_recv] = recovered
    # gleanr sometimes produces a single column of zero values
    # procrustes requires: Input matrices must contain >1 unique points
    # a matrix has no unique points iff max - min == 0.
    if np.ptp(Y) == 0 :
        m2 = np.sum(np.square(X))
        psnr = peak_signal_to_noise_ratio(X, Y, mask)
    else:
        R_orig, R_recv, m2 = procrustes(X, Y)
        psnr = peak_signal_to_noise_ratio(R_orig, R_recv, mask)
    # procrustes produces squared error, not the mean error
    ndim = m * n
    rmse = np.sqrt(m2 / ndim)
    return rmse, psnr


def adjusted_mutual_information_score(X, class_labels):
    X = standardize(X, axis = 0, center = True, scale = False)
    # we know the true clusters
    n_clusters = len(set(class_labels))
    distance_matrix = skmetrics.pairwise.pairwise_distances(X, metric='euclidean')
    model_exact = AgglomerativeClustering(n_clusters = n_clusters, linkage = 'average', metric = 'precomputed')
    class_pred_exact = model_exact.fit_predict(distance_matrix)
    mi_score = skmetrics.mutual_info_score(class_labels, class_pred_exact)
    model_approx = AgglomerativeClustering(n_clusters = n_clusters + 2, linkage = 'average', metric = 'precomputed')
    class_pred_approx = model_approx.fit_predict(distance_matrix)
    adj_mi_score = skmetrics.adjusted_mutual_info_score(class_labels, class_pred_approx)
    return mi_score, adj_mi_score

In [43]:
Ztrue = standardize(Ltrue @ Ftrue.T, axis = 0)
Zrecv = standardize(L @ F.T, axis = 0)
Z_rmse = root_mean_squared_error(Ztrue, Zrecv)
MI, adj_MI = adjusted_mutual_information_score(L, labels)
print(Z_rmse, MI, adj_MI)

0.37990093424192234 0.010971267667462897 0.8388716219387783


In [7]:
np.sqrt(mean_squared_error(Ztrue, Zrecv))

3.2139051370188083

In [24]:
np.std(Zrecv, axis = 1)

array([1.19703702, 0.46935747, 1.15178085, 0.45667759, 0.9831642 ,
       0.80871434, 1.16745559, 0.7248607 , 1.24257061, 0.69198217,
       1.14690554, 0.64302067, 0.76846459, 0.58970589, 1.06593352,
       0.98032922, 0.77475351, 1.43930555, 0.87595861, 0.85350779,
       0.84259448, 0.73196783, 0.56951175, 0.51458457, 1.04448352,
       1.00919331, 0.95256588, 1.05654921, 0.57340505, 0.61926794,
       0.65256001, 1.0834616 , 0.90576071, 1.40320047, 0.98664317,
       1.52139032, 0.78433808, 0.82285788, 0.93670404, 1.12567985,
       0.53789637, 1.35060734, 1.08067861, 1.3644644 , 0.82803441,
       0.55995966, 0.55598742, 1.07370262, 0.87005784, 1.0508336 ,
       1.01728159, 0.75108339, 1.00170651, 0.93323256, 0.86947006,
       1.30825673, 0.31581764, 0.94415917, 0.96022467, 0.95692433,
       0.95644682, 0.4581205 , 0.57114775, 0.70337998, 0.90524565,
       1.10414964, 1.14102587, 1.11916273, 0.83058899, 0.69028712,
       0.67843585, 1.42519032, 0.82882545, 1.02280938, 1.15815

In [9]:
F.shape

(500, 10)